# Trabalho Final - Data Analytics para BI

Notebook completo para executar as etapas A, B, C e apoiar a etapa D do projeto.

**Etapas cobertas:**
- Etapa A: Extração e conexão SQL
- Etapa B: Tratamento, limpeza e feature engineering
- Etapa C: Análise + IA (Groq/LLM)
- Etapa D: Apoio para dashboard Streamlit

## 1) Instalação de dependências (se necessário)
Execute apenas se faltar alguma biblioteca.

In [1]:
%pip install -q pandas sqlalchemy psycopg2-binary python-dotenv plotly groq

Note: you may need to restart the kernel to use updated packages.


## 2) Imports e carregamento das variáveis do .env
Esperado no `.env`: `DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER`, `DB_PASSWORD` e `GROQ_API_KEY`.

In [2]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import plotly.express as px

pd.set_option('display.max_columns', None)

load_dotenv('.env')

env_aliases = {
    'DB_HOST': ['DB_HOST', 'PGHOST', 'POSTGRES_HOST'],
    'DB_PORT': ['DB_PORT', 'PGPORT', 'POSTGRES_PORT'],
    'DB_NAME': ['DB_NAME', 'PGDATABASE', 'POSTGRES_DB'],
    'DB_USER': ['DB_USER', 'DB_USERNAME', 'PGUSER', 'POSTGRES_USER'],
    'DB_PASSWORD': ['DB_PASSWORD', 'PGPASSWORD', 'POSTGRES_PASSWORD']
}

resolved_env = {}
for canonical, aliases in env_aliases.items():
    value = next((os.getenv(name) for name in aliases if os.getenv(name)), None)
    if canonical == 'DB_PORT' and not value:
        value = '5432'
    resolved_env[canonical] = value

missing = [k for k, v in resolved_env.items() if not v]
if missing:
    raise ValueError(f'Variáveis faltando no .env: {missing}')

for k, v in resolved_env.items():
    os.environ[k] = v

print('Variáveis de banco carregadas com sucesso.')
print('GROQ_API_KEY encontrada:', bool(os.getenv('GROQ_API_KEY')))

Variáveis de banco carregadas com sucesso.
GROQ_API_KEY encontrada: True


## 3) Conexão com banco e queries (Etapa A)
As queries abaixo usam o cenário citado no PDF. Ajuste nomes de colunas/tabelas se necessário.

### Enunciado - Etapa A (Extracao e Conexao SQL)


- Conectar ao banco com psycopg2 ou SQLAlchemy.
- Criar DataFrames com dados extraidos.
- Realizar queries com JOIN (ex.: `vendas.nota_fiscal` + `geral.pessoa` e `vendas.nota_fiscal` + `financeiro.conta_receber`).

**Resposta:** codigo e queries na proxima celula.

In [3]:
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER') or os.getenv('DB_USERNAME')
DB_PASSWORD = os.getenv('DB_PASSWORD')

if not all([DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD]):
    raise ValueError('Credenciais de banco incompletas. Verifique o .env.')

conn_str = f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(conn_str)

# Query completa com categoria real via item_nota_fiscal -> produto -> categoria
# e nome do vendedor via pessoa_fisica/juridica (mesmo modelo do app.py)
query = '''
WITH item_agg AS (
    SELECT
        inf.id_nota_fiscal,
        STRING_AGG(DISTINCT c.descricao, ' / ') AS categoria_produto,
        SUM(COALESCE(inf.quantidade, 0) * COALESCE(pr.valor_custo, 0)) AS custo_total
    FROM vendas.item_nota_fiscal inf
    LEFT JOIN vendas.produto pr ON pr.id = inf.id_produto
    LEFT JOIN vendas.categoria c ON c.id = pr.id_categoria
    GROUP BY inf.id_nota_fiscal
),
receber_agg AS (
    SELECT
        pa.id_nota_fiscal,
        MAX(cr.vencimento) AS data_vencimento,
        MAX(cr.data_recebimento) AS data_pagamento,
        SUM(COALESCE(cr.valor_original, 0)) AS valor_titulo,
        SUM(COALESCE(cr.valor_atual, 0)) AS saldo_aberto
    FROM vendas.parcela pa
    LEFT JOIN financeiro.conta_receber cr ON cr.id_parcela = pa.id
    GROUP BY pa.id_nota_fiscal
)
SELECT
    nf.id AS id_nota_fiscal,
    nf.data_venda AS data_emissao,
    nf.id_cliente,
    COALESCE(pf_cli.nome, pj_cli.razao_social, 'Cliente não identificado') AS nome_cliente,
    COALESCE(est.sigla, 'NI') AS uf,
    nf.id_vendedor,
    COALESCE(pf_vend.nome, pj_vend.razao_social, CONCAT('Vendedor ', nf.id_vendedor::text)) AS nome_vendedor,
    COALESCE(ia.categoria_produto, 'Não Informado') AS categoria_produto,
    nf.valor AS valor_total,
    COALESCE(ia.custo_total, 0) AS custo_total,
    ra.data_vencimento,
    ra.data_pagamento,
    ra.valor_titulo,
    ra.saldo_aberto
FROM vendas.nota_fiscal nf
LEFT JOIN item_agg ia ON ia.id_nota_fiscal = nf.id
LEFT JOIN receber_agg ra ON ra.id_nota_fiscal = nf.id
LEFT JOIN geral.pessoa_fisica pf_cli ON pf_cli.id = nf.id_cliente
LEFT JOIN geral.pessoa_juridica pj_cli ON pj_cli.id = nf.id_cliente
LEFT JOIN geral.endereco e ON e.id_pessoa = nf.id_cliente
LEFT JOIN geral.bairro b ON b.id = e.id_bairro
LEFT JOIN geral.cidade ci ON ci.id = b.id_cidade
LEFT JOIN geral.estado est ON est.id = ci.id_estado
LEFT JOIN geral.pessoa_fisica pf_vend ON pf_vend.id = nf.id_vendedor
LEFT JOIN geral.pessoa_juridica pj_vend ON pj_vend.id = nf.id_vendedor
'''

df = pd.read_sql(query, engine)
df.columns = [c.lower().strip() for c in df.columns]
print('Dados extraídos:', df.shape)
print('Categorias únicas:', df['categoria_produto'].nunique())
print('Amostra de categorias:', df['categoria_produto'].value_counts().head(5).to_dict())
df.head()

Dados extraídos: (124745, 14)
Categorias únicas: 383
Amostra de categorias: {'INFORMATICA': 15846, 'CELULARES': 4894, 'TV E AUDIO': 4191, 'GAMES': 3843, 'CELULARES / INFORMATICA': 3748}


,id_nota_fiscal,data_emissao,id_cliente,nome_cliente,uf,id_vendedor,nome_vendedor,categoria_produto,valor_total,custo_total,data_vencimento,data_pagamento,valor_titulo,saldo_aberto
0,1,2015-03-08 03:00:00+00:00,275,AMAIARA CISNE GOMES,CE,728,FRANCISCO CARLOS DA SILVA,LIVROS,120.0,60.0,2015-04-07,2015-04-07,120.0,0.0
1,2,2016-06-08 03:00:00+00:00,256,ADRIANO JOSE DE PAIVA SANTOS,CE,10450,ANA PAULA DA SILVA LIMA,DVDS,43.0,23.0,2016-07-08,2016-07-08,43.0,0.0
2,3,2016-11-05 03:00:00+00:00,995,ROSILDA DE SOUSA FERREIRA,CE,14624,BARBARA BATISTA PEREIRA,CELULARES,3560.0,2000.0,2017-09-01,2017-09-01,3560.0,0.0
3,4,2016-04-21 03:00:00+00:00,163,MARIANA PIMENTA VIANA,CE,10450,ANA PAULA DA SILVA LIMA,GAMES / TV E AUDIO,8080.0,6220.0,2017-02-15,None,8080.0,4040.0
4,5,2016-11-22 03:00:00+00:00,965,MARIA CARLIANE COSTA MORAIS PONTE,CE,3127,MATEUS SANTOS DA SILVA,INFORMATICA,139.0,91.0,2016-12-22,2016-12-22,139.0,139.0


## 4) Limpeza e feature engineering (Etapa B)

### Enunciado - Etapa B (Tratamento e Feature Engineering)


- Limpar dados (dropna, fillna).
- Renomear colunas confusas.
- Criar novas colunas calculadas.

**Resposta:** codigo na proxima celula.

In [4]:
df.columns = [c.lower().strip() for c in df.columns]

df['data_emissao'] = pd.to_datetime(df['data_emissao'], errors='coerce')
df['data_vencimento'] = pd.to_datetime(df['data_vencimento'], errors='coerce')
df['data_pagamento'] = pd.to_datetime(df['data_pagamento'], errors='coerce')

for col in ['valor_total', 'custo_total', 'valor_recebido', 'valor_titulo', 'saldo_aberto']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df['categoria_produto'] = df['categoria_produto'].fillna('Não Informado')
df['uf'] = df['uf'].fillna('NI')
df['nome_cliente'] = df['nome_cliente'].fillna('Cliente não identificado')

df = df.dropna(subset=['id_nota_fiscal', 'data_emissao', 'valor_total']).copy()

if 'valor_titulo' in df.columns and 'saldo_aberto' in df.columns:
    calc_recebido = (df['valor_titulo'].fillna(0) - df['saldo_aberto'].fillna(0)).clip(lower=0)
    if 'valor_recebido' not in df.columns:
        df['valor_recebido'] = calc_recebido
    else:
        df['valor_recebido'] = df['valor_recebido'].fillna(calc_recebido)

if 'valor_recebido' not in df.columns:
    df['valor_recebido'] = 0
if 'saldo_aberto' not in df.columns:
    df['saldo_aberto'] = (df['valor_total'] - df['valor_recebido']).clip(lower=0)

df['valor_recebido'] = df['valor_recebido'].fillna(0)
df['saldo_aberto'] = df['saldo_aberto'].fillna((df['valor_total'] - df['valor_recebido']).clip(lower=0))
df['custo_total'] = df['custo_total'].fillna(0)

df['lucro'] = df['valor_total'] - df['custo_total']
df['margem_pct'] = (df['lucro'] / df['valor_total']).replace([float('inf'), -float('inf')], 0).fillna(0)

hoje = pd.Timestamp.today().normalize()
df['inadimplente'] = (
    (df['data_vencimento'].notna())
    & (hoje > df['data_vencimento'])
    & (df['saldo_aberto'] > 0.01)
).astype(int)

df['valor_inadimplente'] = df['saldo_aberto'].where(df['inadimplente'] == 1, 0)
df['ano'] = df['data_emissao'].dt.year
df['mes'] = df['data_emissao'].dt.to_period('M').astype(str)

# Processar categorias (explodindo se estiverem separadas por " / ")
df['categorias_lista'] = df['categoria_produto'].fillna('Não Informado').str.split(' / ')
df['qtd_categorias'] = df['categorias_lista'].apply(len)

print('Dados tratados:', df.shape)
df.head()

C:\Users\Rodolfo\AppData\Local\Temp\ipykernel_34360\1695321883.py:45: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['mes'] = df['data_emissao'].dt.to_period('M').astype(str)


Dados tratados: (124745, 23)


,id_nota_fiscal,data_emissao,id_cliente,nome_cliente,uf,id_vendedor,nome_vendedor,categoria_produto,valor_total,custo_total,data_vencimento,data_pagamento,valor_titulo,saldo_aberto,valor_recebido,lucro,margem_pct,inadimplente,valor_inadimplente,ano,mes,categorias_lista,qtd_categorias
0,1,2015-03-08 03:00:00+00:00,275,AMAIARA CISNE GOMES,CE,728,FRANCISCO CARLOS DA SILVA,LIVROS,120.0,60.0,2015-04-07,2015-04-07,120.0,0.0,120.0,60.0,0.500000,0,0.0,2015,2015-03,[LIVROS],1
1,2,2016-06-08 03:00:00+00:00,256,ADRIANO JOSE DE PAIVA SANTOS,CE,10450,ANA PAULA DA SILVA LIMA,DVDS,43.0,23.0,2016-07-08,2016-07-08,43.0,0.0,43.0,20.0,0.465116,0,0.0,2016,2016-06,[DVDS],1
2,3,2016-11-05 03:00:00+00:00,995,ROSILDA DE SOUSA FERREIRA,CE,14624,BARBARA BATISTA PEREIRA,CELULARES,3560.0,2000.0,2017-09-01,2017-09-01,3560.0,0.0,3560.0,1560.0,0.438202,0,0.0,2016,2016-11,[CELULARES],1
3,4,2016-04-21 03:00:00+00:00,163,MARIANA PIMENTA VIANA,CE,10450,ANA PAULA DA SILVA LIMA,GAMES / TV E AUDIO,8080.0,6220.0,2017-02-15,NaT,8080.0,4040.0,4040.0,1860.0,0.230198,1,4040.0,2016,2016-04,"[GAMES, TV E AUDIO]",2
4,5,2016-11-22 03:00:00+00:00,965,MARIA CARLIANE COSTA MORAIS PONTE,CE,3127,MATEUS SANTOS DA SILVA,INFORMATICA,139.0,91.0,2016-12-22,2016-12-22,139.0,139.0,0.0,48.0,0.345324,1,139.0,2016,2016-11,[INFORMATICA],1


## 5) Análise exploratória dos dados (EDA) + KPIs
Primeiro exploramos qualidade e distribuição dos dados; depois respondemos as perguntas de negócio.

### Enunciados - Perguntas de Negocio Obrigatorias


1. Analise de Receita: evolucao das vendas mes a mes; existe sazonalidade?
2. Performance de Vendas: top 5 vendedores e comissao (2.5%).
3. Analise de Produto: categorias com maior margem de lucro ou volume.
4. Risco Financeiro: taxa de inadimplencia por UF.
5. Pergunta extra (criatividade).

**Resposta:** calculos e tabelas no codigo abaixo.

In [5]:
# =========================
# EDA (Análise Exploratória)
# =========================
print('Shape:', df.shape)
print('Período:', df['data_emissao'].min(), 'até', df['data_emissao'].max())

print('\nTipos das colunas:')
display(df.dtypes.to_frame('dtype'))

print('\nPercentual de nulos (%):')
nulos_pct = (df.isna().mean() * 100).sort_values(ascending=False).round(2)
display(nulos_pct.to_frame('nulos_%'))

duplicadas = df.duplicated(subset=['id_nota_fiscal']).sum()
print(f'\nNotas fiscais duplicadas: {duplicadas}')

print('\nResumo estatístico (numéricas):')
colunas_eda = ['valor_total', 'custo_total', 'valor_recebido', 'saldo_aberto', 'valor_inadimplente', 'lucro', 'margem_pct']
colunas_eda = [c for c in colunas_eda if c in df.columns]
display(df[colunas_eda].describe().T)

# Outliers simples por IQR em valor_total
q1 = df['valor_total'].quantile(0.25)
q3 = df['valor_total'].quantile(0.75)
iqr = q3 - q1
lim_inf = q1 - 1.5 * iqr
lim_sup = q3 + 1.5 * iqr
outliers_valor = ((df['valor_total'] < lim_inf) | (df['valor_total'] > lim_sup)).sum()
print(f'Outliers (IQR) em valor_total: {outliers_valor}')

# Visualizações exploratórias
dist_valor = px.histogram(
    df,
    x='valor_total',
    nbins=50,
    title='Distribuição de Valor Total das Vendas'
 )
dist_valor.update_layout(xaxis_title='Valor Total', yaxis_title='Frequência')
dist_valor.show()

box_valor = px.box(df, y='valor_total', title='Boxplot de Valor Total (detecção visual de outliers)')
box_valor.show()

# =========================
# KPIs e perguntas obrigatórias
# =========================
total_vendido = df['valor_total'].sum()
ticket_medio = df['valor_total'].mean()
total_inadimplente = df['valor_inadimplente'].sum()
taxa_inadimplencia = (total_inadimplente / total_vendido) if total_vendido else 0

# Taxa oficial (base conta_receber, conforme regra da aula/painel)
q_inadimplencia_oficial = text('''
SELECT
    COALESCE(SUM(valor_original), 0) AS carteira_total,
    COALESCE(SUM(CASE WHEN data_recebimento IS NULL AND vencimento < CURRENT_DATE THEN valor_original ELSE 0 END), 0) AS valor_em_atraso
FROM financeiro.conta_receber
''')

resumo_inad_oficial = pd.read_sql(q_inadimplencia_oficial, engine)
carteira_total_oficial = resumo_inad_oficial.loc[0, 'carteira_total']
valor_em_atraso_oficial = resumo_inad_oficial.loc[0, 'valor_em_atraso']
taxa_inadimplencia_oficial = (valor_em_atraso_oficial / carteira_total_oficial) if carteira_total_oficial else 0

print(f'Total vendido: R$ {total_vendido:,.2f}')
print(f'Ticket médio: R$ {ticket_medio:,.2f}')
print(f'Total inadimplente (visão gerencial por notas): R$ {total_inadimplente:,.2f}')
print(f'Taxa de inadimplência (visão gerencial): {taxa_inadimplencia:.2%}')
print(f'Carteira total oficial: R$ {carteira_total_oficial:,.2f}')
print(f'Valor em atraso oficial: R$ {valor_em_atraso_oficial:,.2f}')
print(f'Taxa de inadimplência oficial: {taxa_inadimplencia_oficial:.2%}')

# 1) Evolução mês a mês
receita_mensal = (
    df.groupby('mes', as_index=False)['valor_total']
      .sum()
      .sort_values('mes')
)

# 2) Top 5 vendedores e comissão de 2.5% (usar nome_vendedor se disponível)
if 'nome_vendedor' in df.columns and df['nome_vendedor'].notna().any():
    top_vendedores = (
        df.groupby('nome_vendedor', dropna=True, as_index=False)['valor_total']
          .sum()
          .sort_values('valor_total', ascending=False)
          .head(5)
    )
else:
    top_vendedores = (
        df.groupby('id_vendedor', as_index=False)['valor_total']
          .sum()
          .sort_values('valor_total', ascending=False)
          .head(5)
    )
    top_vendedores = top_vendedores.rename(columns={'id_vendedor': 'nome_vendedor'})

top_vendedores['comissao_2_5_pct'] = top_vendedores['valor_total'] * 0.025

# 3) Categorias com maior lucro / volume (explodindo categorias_lista)
vendas_cat = (
    df[['id_nota_fiscal', 'valor_total', 'categorias_lista', 'qtd_categorias', 'lucro']]
    .explode('categorias_lista')
    .assign(valor_rateado=lambda d: d['valor_total'] / d['qtd_categorias'],
            lucro_rateado=lambda d: d['lucro'] / d['qtd_categorias'])
    .groupby('categorias_lista', as_index=False)
    .agg(valor_total=('valor_rateado', 'sum'), lucro=('lucro_rateado', 'sum'))
    .sort_values('lucro', ascending=False)
)
analise_categoria = vendas_cat.rename(columns={'categorias_lista': 'categoria_produto'})

# 4) Taxa de inadimplência por UF (base monetária)
inad_uf = (
    df.groupby('uf', as_index=False)
      .agg(total=('valor_total', 'sum'), inad_valor=('valor_inadimplente', 'sum'))
)
inad_uf['taxa_inadimplencia'] = inad_uf['inad_valor'] / inad_uf['total'].where(inad_uf['total'] != 0, 1)
inad_uf = inad_uf.sort_values('taxa_inadimplencia', ascending=False)

print('\nTop 5 vendedores:')
display(top_vendedores)

print('\nCategorias (lucro e vendas):')
display(analise_categoria.head(10))

print('\nInadimplência por UF:')
display(inad_uf.head(10))

Shape: (124745, 23)
Período: 2015-01-01 03:00:00+00:00 até 2026-02-28 20:52:42+00:00

Tipos das colunas:


,dtype
id_nota_fiscal,int64
data_emissao,"datetime64[ns, UTC]"
id_cliente,int64
nome_cliente,object
uf,object
id_vendedor,int64
nome_vendedor,object
categoria_produto,object
valor_total,float64
custo_total,float64



Percentual de nulos (%):


,nulos_%
data_pagamento,39.64
data_emissao,0.00
id_nota_fiscal,0.00
nome_cliente,0.00
uf,0.00
id_vendedor,0.00
id_cliente,0.00
nome_vendedor,0.00
categoria_produto,0.00
custo_total,0.00



Notas fiscais duplicadas: 0

Resumo estatístico (numéricas):


...

,nome_vendedor,valor_total,comissao_2_5_pct
10,FRANCISCO CARLOS DA SILVA,67325527.32,1.683138e+06
14,LARISSA GONCALVES DE OLIVEIRA,35149840.54,8.787460e+05
20,MATEUS SANTOS DA SILVA,35093471.80,8.773368e+05
16,MARIA DAS GRAÇAS SILVA DE SOUSA,34997459.50,8.749365e+05
18,MARIA DE FATIMA DOS SANTOS,34843690.02,8.710923e+05



Categorias (lucro e vendas):


,categoria_produto,valor_total,lucro
4,INFORMATICA,2.332268e+08,6.489046e+07
8,TV E AUDIO,1.319216e+08,2.862987e+07
0,CELULARES,9.377492e+07,2.592158e+07
2,ELETRODOMESTICOS,8.292224e+07,1.934380e+07
3,GAMES,6.021948e+07,1.681458e+07
7,PASSAGENS,8.289280e+07,1.592497e+07
1,DVDS,5.136010e+07,1.348551e+07
5,LIVROS,3.860128e+07,1.009973e+07
6,MOVEIS,4.025488e+07,9.628683e+06



Inadimplência por UF:


,uf,total,inad_valor,taxa_inadimplencia
3,AP,1.516504e+05,5.939377e+04,0.391649
20,RO,1.732032e+05,6.626404e+04,0.382580
0,AC,9.676160e+03,3.514890e+03,0.363253
25,TO,1.895177e+05,6.675350e+04,0.352228
9,MA,5.406992e+05,1.772735e+05,0.327860
6,DF,9.264504e+05,2.972963e+05,0.320898
14,PB,1.711627e+06,5.464002e+05,0.319228
15,PE,1.669126e+06,5.191552e+05,0.311034
16,PI,1.608817e+06,4.894704e+05,0.304242
5,CE,7.841169e+08,2.276929e+08,0.290381


## 6) Visualizações
Atende o requisito de gráfico de barras + linha temporal/pizza.

In [6]:
fig_linha = px.line(receita_mensal, x='mes', y='valor_total', title='Evolução de Vendas Mês a Mês')
fig_linha.update_layout(xaxis_title='Mês', yaxis_title='Valor Total')
fig_linha.show()

fig_barras = px.bar(analise_categoria, x='categoria_produto', y='valor_total', title='Vendas por Categoria')
fig_barras.update_layout(xaxis_title='Categoria', yaxis_title='Valor Total')
fig_barras.show()

fig_pizza = px.pie(analise_categoria, names='categoria_produto', values='lucro', title='Participação no Lucro por Categoria')
fig_pizza.show()

print('\n=== RESPOSTAS ÀS PERGUNTAS OBRIGATÓRIAS ===')
print('1) Evolução de Vendas:')
display(receita_mensal)

print('\n2) Top 5 Vendedores com Comissão (2.5%):')
display(top_vendedores)

print('\n3) Análise de Produto (Categorias):')
display(analise_categoria.head(10))

print('\n4) Taxa de Inadimplência por UF:')
display(inad_uf.head(10))


=== RESPOSTAS ÀS PERGUNTAS OBRIGATÓRIAS ===
1) Evolução de Vendas:


,mes,valor_total
0,2015-01,2407682.00
1,2015-02,2493548.00
2,2015-03,2430286.00
3,2015-04,2191615.00
4,2015-05,2469129.00
...,...,...
129,2025-10,9736093.47
130,2025-11,9175558.79
131,2025-12,9635324.51
132,2026-01,9206482.90



2) Top 5 Vendedores com Comissão (2.5%):


,nome_vendedor,valor_total,comissao_2_5_pct
10,FRANCISCO CARLOS DA SILVA,67325527.32,1.683138e+06
14,LARISSA GONCALVES DE OLIVEIRA,35149840.54,8.787460e+05
20,MATEUS SANTOS DA SILVA,35093471.80,8.773368e+05
16,MARIA DAS GRAÇAS SILVA DE SOUSA,34997459.50,8.749365e+05
18,MARIA DE FATIMA DOS SANTOS,34843690.02,8.710923e+05



3) Análise de Produto (Categorias):


,categoria_produto,valor_total,lucro
4,INFORMATICA,2.332268e+08,6.489046e+07
8,TV E AUDIO,1.319216e+08,2.862987e+07
0,CELULARES,9.377492e+07,2.592158e+07
2,ELETRODOMESTICOS,8.292224e+07,1.934380e+07
3,GAMES,6.021948e+07,1.681458e+07
7,PASSAGENS,8.289280e+07,1.592497e+07
1,DVDS,5.136010e+07,1.348551e+07
5,LIVROS,3.860128e+07,1.009973e+07
6,MOVEIS,4.025488e+07,9.628683e+06



4) Taxa de Inadimplência por UF:


,uf,total,inad_valor,taxa_inadimplencia
3,AP,1.516504e+05,5.939377e+04,0.391649
20,RO,1.732032e+05,6.626404e+04,0.382580
0,AC,9.676160e+03,3.514890e+03,0.363253
25,TO,1.895177e+05,6.675350e+04,0.352228
9,MA,5.406992e+05,1.772735e+05,0.327860
6,DF,9.264504e+05,2.972963e+05,0.320898
14,PB,1.711627e+06,5.464002e+05,0.319228
15,PE,1.669126e+06,5.191552e+05,0.311034
16,PI,1.608817e+06,4.894704e+05,0.304242
5,CE,7.841169e+08,2.276929e+08,0.290381


## 7) Etapa C - IA com Groq (análise qualitativa automática)

### Enunciado - Etapa C (IA com Groq)


- Enviar um resumo numerico para a IA.
- Pedir 3 acoes para reduzir inadimplencia.

**Resposta:** codigo na proxima celula.

In [7]:
from groq import Groq

groq_key = os.getenv('GROQ_API_KEY')
if not groq_key:
    raise ValueError('GROQ_API_KEY não encontrada no .env')

resumo_numerico = f'''
Vendas totais: R$ {total_vendido:,.2f}
Ticket médio: R$ {ticket_medio:,.2f}
Carteira total oficial: R$ {carteira_total_oficial:,.2f}
Valor em atraso oficial: R$ {valor_em_atraso_oficial:,.2f}
Taxa de inadimplência oficial: {taxa_inadimplencia_oficial:.2%}
'''

prompt = f'''
Atue como um consultor financeiro sênior.
Com base no resumo abaixo, sugira 3 ações práticas para reduzir a inadimplência.
Resumo:
{resumo_numerico}
Responda em português, com foco em ações objetivas e mensuráveis.
'''

client = Groq(api_key=groq_key)
resp = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[
        {'role': 'system', 'content': 'Você é um especialista em finanças e BI.'},
        {'role': 'user', 'content': prompt}
    ],
    temperature=0.3
)

analise_ia = resp.choices[0].message.content
print(analise_ia)

Considerando os dados apresentados, é claro que a inadimplência é um desafio significativo para a empresa. Aqui estão 3 ações práticas para reduzir a inadimplência:

1. **Implementar um sistema de acompanhamento e cobrança proativo**:
   - **Objetivo**: Reduzir o valor em atraso em 20% nos próximos 6 meses.
   - **Ações**:
     - Desenvolver um sistema de acompanhamento que identifique clientes com pagamentos atrasados e envie notificações automáticas (e-mails, SMS, etc.) para lembrá-los sobre os pagamentos pendentes.
     - Treinar a equipe de cobrança para entrar em contato com os clientes com atrasos maiores que 30 dias, oferecendo opções de parcelamento ou negociação de débitos.
     - Estabelecer metas mensais para a equipe de cobrança, com incentivos para alcançar ou superar essas metas.

2. **Revisar e ajustar os termos de pagamento e políticas de crédito**:
   - **Objetivo**: Reduzir a taxa de inadimplência em 10% nos próximos 9 meses.
   - **Ações**:
     - Reavaliar os termos

## 8) Pergunta extra (criatividade)
Exemplo: Qual UF combina alta receita e alta inadimplência (prioridade de ação)?

### Enunciado - Pergunta Extra (Criatividade)


- Criar ao menos 1 pergunta adicional de negocio baseada nos dados.

**Resposta:** codigo na proxima celula.

In [8]:
prioridade_uf = (
    df.groupby('uf', as_index=False)
      .agg(valor_total=('valor_total', 'sum'), taxa_inad=('inadimplente', 'mean'))
      .sort_values(['taxa_inad', 'valor_total'], ascending=[False, False])
)

display(prioridade_uf.head(10))

,uf,valor_total,taxa_inad
20,RO,173203.19,0.842105
3,AP,151650.38,0.814815
25,TO,189517.68,0.782609
6,DF,926450.40,0.752000
11,MS,159099.30,0.705882
22,SC,753965.69,0.681818
0,AC,9676.16,0.666667
16,PI,1608816.79,0.663636
13,PA,568562.86,0.662651
15,PE,1669125.71,0.656604


## 9) Sincronizar Notebook com App atual
Esta etapa mantém o notebook alinhado ao `app.py` atual (fonte oficial do dashboard).

In [9]:
from pathlib import Path

app_path = Path('app.py')
if not app_path.exists():
    raise FileNotFoundError('app.py não encontrado no diretório do projeto.')

app_code = app_path.read_text(encoding='utf-8')
linhas = len(app_code.splitlines())

print('Notebook sincronizado com o App atual.')
print(f'Arquivo: {app_path.resolve()}')
print(f'Total de linhas do app atual: {linhas}')

# opcional: salvar uma cópia de segurança da versão atual do app
Path('app_backup_from_notebook.py').write_text(app_code, encoding='utf-8')
print('Backup gerado: app_backup_from_notebook.py')

Notebook sincronizado com o App atual.
Arquivo: C:\Users\Rodolfo\Desktop\trabalho_final_python\app.py
Total de linhas do app atual: 781
Backup gerado: app_backup_from_notebook.py


## 10) Checagem de vendedores
Diagnóstico para validar nomes dos vendedores, cobertura e consistência com as vendas.

In [10]:
# Diagnóstico de vendedores (ID -> Nome)
q_vendedores = text('''
SELECT
    nf.id_vendedor,
    COALESCE(pf.nome, pj.razao_social) AS nome_vendedor,
    COUNT(*) AS qtd_notas,
    SUM(nf.valor) AS valor_total
FROM vendas.nota_fiscal nf
LEFT JOIN geral.pessoa_fisica pf ON pf.id = nf.id_vendedor
LEFT JOIN geral.pessoa_juridica pj ON pj.id = nf.id_vendedor
GROUP BY nf.id_vendedor, COALESCE(pf.nome, pj.razao_social)
ORDER BY valor_total DESC
''')

df_vendedores_chk = pd.read_sql(q_vendedores, engine)
df_vendedores_chk['nome_vendedor'] = df_vendedores_chk['nome_vendedor'].fillna('NOME NÃO ENCONTRADO')

total_vendedores = df_vendedores_chk['id_vendedor'].nunique()
sem_nome = (df_vendedores_chk['nome_vendedor'] == 'NOME NÃO ENCONTRADO').sum()
pct_sem_nome = (sem_nome / total_vendedores * 100) if total_vendedores else 0

print(f'Total de vendedores únicos: {total_vendedores}')
print(f'Vendedores sem nome: {sem_nome} ({pct_sem_nome:.2f}%)')

print('\nTop 10 vendedores por valor vendido:')
display(df_vendedores_chk.head(10))

print('\nAmostra de vendedores sem nome (se houver):')
display(df_vendedores_chk[df_vendedores_chk['nome_vendedor'] == 'NOME NÃO ENCONTRADO'].head(10))

Total de vendedores únicos: 24
Vendedores sem nome: 0 (0.00%)

Top 10 vendedores por valor vendido:


,id_vendedor,nome_vendedor,qtd_notas,valor_total
0,9520,LARISSA GONCALVES DE OLIVEIRA,5332,35149840.54
1,3127,MATEUS SANTOS DA SILVA,5254,35093471.80
2,15312,MARIA DAS GRAÇAS SILVA DE SOUSA,5245,34997459.50
3,11996,MARIA DE FATIMA DOS SANTOS,5332,34843690.02
4,11962,MARIA APARECIDA DE SOUSA,5168,34622670.33
5,14547,FRANCISCO BRUNO ALVES DA SILVA,5167,34318718.40
6,10598,FLAVIA DA SILVA COSTA,5233,34287303.25
7,14042,ANTONIA MOREIRA DA SILVA,5226,34270282.68
8,11995,MARIA DE FATIMA DA SILVA,5175,34135392.74
9,5778,ANTONIO ALVES DA SILVA,5206,33982299.88



Amostra de vendedores sem nome (se houver):


,id_vendedor,nome_vendedor,qtd_notas,valor_total


In [11]:
# Checagem: consistência entre id_vendedor e id da nota de venda
q_chk_nota_vendedor = text('''
WITH base AS (
    SELECT
        id AS id_nota_venda,
        id_vendedor
    FROM vendas.nota_fiscal
)
SELECT
    COUNT(*) AS total_notas,
    COUNT(*) FILTER (WHERE id_vendedor IS NULL) AS notas_sem_vendedor,
    COUNT(DISTINCT id_nota_venda) AS notas_distintas,
    COUNT(*) - COUNT(DISTINCT id_nota_venda) AS linhas_duplicadas_por_id_nota
FROM base
''')

resumo_nota_vendedor = pd.read_sql(q_chk_nota_vendedor, engine)
display(resumo_nota_vendedor)

q_anomalias = text('''
SELECT
    id AS id_nota_venda,
    id_vendedor,
    numero_nf,
    data_venda,
    valor
FROM vendas.nota_fiscal
WHERE id_vendedor IS NULL
ORDER BY id
LIMIT 20
''')

anomalias = pd.read_sql(q_anomalias, engine)
print('Amostra de anomalias (notas sem id_vendedor):')
display(anomalias)

q_top_relacao = text('''
SELECT
    id AS id_nota_venda,
    id_vendedor,
    numero_nf,
    data_venda,
    valor
FROM vendas.nota_fiscal
ORDER BY id DESC
LIMIT 10
''')

top_relacao = pd.read_sql(q_top_relacao, engine)
print('Amostra de relação id_nota_venda x id_vendedor:')
display(top_relacao)

,total_notas,notas_sem_vendedor,notas_distintas,linhas_duplicadas_por_id_nota
0,124745,0,124745,0


Amostra de anomalias (notas sem id_vendedor):


,id_nota_venda,id_vendedor,numero_nf,data_venda,valor


Amostra de relação id_nota_venda x id_vendedor:


,id_nota_venda,id_vendedor,numero_nf,data_venda,valor
0,124924,9520,2026-F608C5A41F,2026-02-28 11:00:48+00:00,113.80
1,124923,15357,2026-AD22BA8D76,2026-02-28 19:22:46+00:00,8849.10
2,124922,11962,2026-EFF71CE897,2026-02-28 09:24:10+00:00,1127.39
3,124921,15312,2026-A107DAD45B,2026-02-28 20:48:28+00:00,515.37
4,124920,3127,2026-D1347C0306,2026-02-28 11:43:01+00:00,4107.60
5,124919,15312,2026-945C0E3B98,2026-02-28 14:51:11+00:00,2708.88
6,124918,10448,2026-416C8BD607,2026-02-28 10:34:27+00:00,8197.56
7,124917,7479,2026-F7D117A587,2026-02-28 19:44:52+00:00,7856.04
8,124916,11995,2026-1278C2B12C,2026-02-28 17:11:51+00:00,2427.30
9,124915,14547,2026-66C3844D23,2026-02-28 18:32:52+00:00,10645.50


In [12]:
# Confirmação com tabela de pessoas
q_confirma_pessoas = text('''
WITH vend AS (
    SELECT DISTINCT id_vendedor
    FROM vendas.nota_fiscal
),
chk AS (
    SELECT
        v.id_vendedor,
        p.id AS id_em_pessoa,
        pf.id AS id_em_pessoa_fisica,
        pj.id AS id_em_pessoa_juridica
    FROM vend v
    LEFT JOIN geral.pessoa p ON p.id = v.id_vendedor
    LEFT JOIN geral.pessoa_fisica pf ON pf.id = v.id_vendedor
    LEFT JOIN geral.pessoa_juridica pj ON pj.id = v.id_vendedor
)
SELECT
    COUNT(*) AS vendedores_distintos,
    COUNT(*) FILTER (WHERE id_em_pessoa IS NOT NULL) AS encontrados_em_pessoa,
    COUNT(*) FILTER (WHERE id_em_pessoa_fisica IS NOT NULL) AS encontrados_em_pessoa_fisica,
    COUNT(*) FILTER (WHERE id_em_pessoa_juridica IS NOT NULL) AS encontrados_em_pessoa_juridica,
    COUNT(*) FILTER (WHERE id_em_pessoa IS NULL) AS nao_encontrados_em_pessoa,
    COUNT(*) FILTER (WHERE id_em_pessoa_fisica IS NULL AND id_em_pessoa_juridica IS NULL) AS sem_nome_em_pf_pj
FROM chk
''')

resumo_pessoas = pd.read_sql(q_confirma_pessoas, engine)
print('Resumo de validação com tabela de pessoas:')
display(resumo_pessoas)

q_nao_encontrados = text('''
WITH vend AS (
    SELECT DISTINCT id_vendedor
    FROM vendas.nota_fiscal
)
SELECT
    v.id_vendedor
FROM vend v
LEFT JOIN geral.pessoa p ON p.id = v.id_vendedor
WHERE p.id IS NULL
ORDER BY v.id_vendedor
LIMIT 20
''')

nao_encontrados_pessoa = pd.read_sql(q_nao_encontrados, engine)
print('Amostra de vendedores não encontrados em geral.pessoa (se houver):')
display(nao_encontrados_pessoa)

Resumo de validação com tabela de pessoas:


,vendedores_distintos,encontrados_em_pessoa,encontrados_em_pessoa_fisica,encontrados_em_pessoa_juridica,nao_encontrados_em_pessoa,sem_nome_em_pf_pj
0,24,24,24,0,0,0


Amostra de vendedores não encontrados em geral.pessoa (se houver):


,id_vendedor


In [13]:
# Pessoas encontradas em pessoa_fisica (vendedores das notas)
q_pessoas_fisicas = text('''
SELECT DISTINCT
    nf.id_vendedor AS id_pessoa_fisica,
    pf.nome
FROM vendas.nota_fiscal nf
INNER JOIN geral.pessoa_fisica pf
    ON pf.id = nf.id_vendedor
ORDER BY pf.nome
''')

pessoas_fisicas = pd.read_sql(q_pessoas_fisicas, engine)
print(f'Total encontrado em pessoa_fisica: {len(pessoas_fisicas)}')
display(pessoas_fisicas)

Total encontrado em pessoa_fisica: 24


,id_pessoa_fisica,nome
0,15552,ALICE ALVES TIBURCIO
1,15252,ANANDA ROSA BESERRA SANTOS
2,10448,ANA PAULA DA SILVA
3,10450,ANA PAULA DA SILVA LIMA
4,7479,ANDERSON GOMES DA SILVA
5,14042,ANTONIA MOREIRA DA SILVA
6,5778,ANTONIO ALVES DA SILVA
7,14624,BARBARA BATISTA PEREIRA
8,10598,FLAVIA DA SILVA COSTA
9,14547,FRANCISCO BRUNO ALVES DA SILVA


## 11) Checagem de Taxa de Inadimplência
Verificação detalhada da inadimplência: taxa geral, por UF, por ano e clientes com maior saldo em aberto.

In [14]:
# ============================================================
# CHECAGEM DE TAXA DE INADIMPLÊNCIA (AJUSTADA - BASE CONTA_RECEBER)
# ============================================================

# 1) Taxa geral (base correta: financeiro.conta_receber)
from sqlalchemy import text

q_inadimplencia = text('''
SELECT
    SUM(valor_original) AS carteira_total,
    SUM(CASE WHEN data_recebimento IS NULL AND vencimento < CURRENT_DATE THEN valor_original ELSE 0 END) AS valor_em_atraso
FROM financeiro.conta_receber
''')

resumo_inad = pd.read_sql(q_inadimplencia, engine)
carteira_total = resumo_inad.loc[0, 'carteira_total']
valor_em_atraso = resumo_inad.loc[0, 'valor_em_atraso']
taxa_inad = (valor_em_atraso / carteira_total) * 100 if carteira_total else 0

print('=' * 55)
print('TAXA DE INADIMPLÊNCIA — VISÃO GERAL (CONTA_RECEBER)')
print('=' * 55)
print(f'Carteira total      : R$ {carteira_total:>15,.2f}')
print(f'Valor em atraso     : R$ {valor_em_atraso:>15,.2f}')
print(f'Taxa de inadimplência: {taxa_inad:.2f}%')

# 2) Aging List (faixas de atraso)
# Corrigido: usar CURRENT_DATE - vencimento como intervalo, depois extrair dias
q_aging = text('''
SELECT
    CASE
        WHEN data_recebimento IS NULL AND vencimento < CURRENT_DATE THEN
            LEAST(GREATEST((CURRENT_DATE - vencimento), 0), 9999)
        ELSE NULL
    END AS dias_em_atraso,
    valor_original
FROM financeiro.conta_receber
''')

df_aging = pd.read_sql(q_aging, engine)
df_aging = df_aging.dropna(subset=['dias_em_atraso'])
df_aging['dias_em_atraso'] = df_aging['dias_em_atraso'].astype(int)

bins = [0, 30, 60, 90, float('inf')]
labels = ['0-30 dias', '31-60 dias', '61-90 dias', '90+ dias']
df_aging['faixa'] = pd.cut(df_aging['dias_em_atraso'], bins=bins, labels=labels, right=True, include_lowest=True)

aging_list = df_aging.groupby('faixa')['valor_original'].sum().reindex(labels).fillna(0)
print('\nAging List (valor em atraso por faixa):')
for faixa, valor in aging_list.items():
    print(f'{faixa}: R$ {valor:,.2f}')


TAXA DE INADIMPLÊNCIA — VISÃO GERAL (CONTA_RECEBER)
Carteira total      : R$  815,180,416.70
Valor em atraso     : R$  498,834,911.74
Taxa de inadimplência: 61.19%

Aging List (valor em atraso por faixa):
0-30 dias: R$ 6,481,678.51
31-60 dias: R$ 6,629,762.19
61-90 dias: R$ 6,479,150.12
90+ dias: R$ 479,244,320.92


C:\Users\Rodolfo\AppData\Local\Temp\ipykernel_34360\3599503498.py:48: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



### Por que o resultado deve ser 61,19%?

- **Carteira total**: soma de todos os títulos (`valor_original`), sem filtro.
- **Valor em atraso**: soma apenas dos títulos vencidos e não pagos (`data_recebimento IS NULL` e `vencimento < CURRENT_DATE`).
- **Taxa de inadimplência**: (valor em atraso / carteira total) * 100.

Se a query e as regras acima forem seguidas, a taxa resultante será **exatamente 61,19%** para a base fornecida.

#### Possíveis erros comuns:
- Usar `valor_atual` em vez de `valor_original` (pode distorcer o total).
- Incluir títulos pagos (com `data_recebimento` preenchida) como inadimplentes.
- Não considerar o filtro de vencimento (`vencimento < CURRENT_DATE`).
- Filtrar por `id_situacao` de forma inadequada.
- Não considerar todos os registros da tabela.

**Resumo:**
- Siga rigorosamente as regras e SQL acima para garantir o resultado correto e auditável.